In [2]:
!python --version

Python 3.11.5


NHTSA is the official source for the “what goes wrong with this car” part of Mile Zero. Matching a vibe to a car is only half the job. The other half is helping someone buy informed, and that needs real information about known problems.

Why NHTSA works for that:

It’s official and trustworthy. It’s the U.S. government’s vehicle safety agency. Owners file complaints there, and each complaint is tagged with the part that failed (engine, brakes, electrical, and so on).
It’s free and needs no key. Anyone can rerun your code, which the course’s reproducibility requirement asks for.
It avoids scraping. We said earlier we’d stay away from forum scraping because of terms-of-service and fragility problems. This is a public API meant to be used.

In [1]:
#Installing packages for working with data
import requests, pandas as pd
print(requests.__version__, pd.__version__)

2.34.2 3.0.6


In [ ]:
url = "https://api.nhtsa.gov/complaints/complaintsByVehicle"
params = {"make": "ACURA", "model": "RSX", "modelYear": 2002}

response = requests.get(url, params=params, timeout=30)
print(response.status_code)

data = response.json() #Converts the response to a Python dictionary
print(list(data.keys()))

200
['count', 'message', 'results']


In [ ]:
print(data["count"])
print(data["message"])
first = data["results"][0]
print(list(first.keys()))
#93 Users filed cmoplaints about 2002 Acura RSX vehicles. 

93
Results returned successfully
['odiNumber', 'manufacturer', 'crash', 'fire', 'numberOfInjuries', 'numberOfDeaths', 'dateOfIncident', 'dateComplaintFiled', 'vin', 'components', 'summary', 'products']


In [ ]:
#Odometer fraud, but labelled as electrical system. 
print(first["components"])
print(first["crash"], first["fire"])
print(first["summary"])

ELECTRICAL SYSTEM
False False
Odometer Fraud. The contact owned a 2002 Acura SRX. The contact stated that the vehicle was sold to a friend in a private sale. The new owner then attempted to sell the vehicle to another person, and upon obtaining the Carfax report, it was discovered that there was a mileage discrepancy. At the time of purchase, the vehicle mileage was approximately 115,000, and at the time of registration, it was discovered that the mileage was 213,288.


In [6]:
for c in data["results"][:10]:
    print(c["components"], "|", c["summary"][:120])
    print()

ELECTRICAL SYSTEM | Odometer Fraud. The contact owned a 2002 Acura SRX. The contact stated that the vehicle was sold to a friend in a privat

STEERING,ELECTRICAL SYSTEM | Duralast 12volt battery post are smaller than they should be. It cause the battery terminal to came loose after a hard b

SEATS | The contact owns a 2022 Acura RSX. The contact stated that while the vehicle was parked, the contact became aware that t

POWER TRAIN | The car stopped at an traffic intersection when making a left turn. Fortunately, the traffic was light. Later, a broken 

AIR BAGS,FORWARD COLLISION AVOIDANCE | THE LIGHT FOR MY AIR BAG IS ON AND THE PASSENGER AIR BAG LIGHT FLICKERS ON TOO.  THE CRUSE CONTROL DOES NOT WORK.

FUEL/PROPULSION SYSTEM | MY CAR HAD A STRONG GAS SMELL INSIDE AND OUTSIDE VEHICLE WHENEVER I DROVE IT. I LOOKED AND FOUND THAT THE GAS PUMP PLAST

FUEL/PROPULSION SYSTEM | WHILE DRIVING I STARTED TO SMELL FUEL IN THE PASSENGER COMPARTMENT. UPON INSPECTION OF THE FUEL PUMP, I DISCOVERED 

In [7]:
df = pd.DataFrame(data["results"])
print(df.shape)
print(df["components"].value_counts().head(10))

(93, 12)
components
POWER TRAIN                  24
ENGINE AND ENGINE COOLING    11
AIR BAGS                      9
STRUCTURE                     4
EXTERIOR LIGHTING             3
SERVICE BRAKES, HYDRAULIC     3
SEATS                         2
FUEL/PROPULSION SYSTEM        2
STEERING                      2
AIR BAGS,SEAT BELTS           2
Name: count, dtype: int64


In [8]:
import re

df["component_list"] = df["components"].apply(lambda s: re.split(r",(?! )", s))
print(df["component_list"].explode().value_counts().head(10))

component_list
POWER TRAIN                  30
ENGINE AND ENGINE COOLING    20
AIR BAGS                     14
STRUCTURE                     6
ELECTRICAL SYSTEM             4
EXTERIOR LIGHTING             4
SERVICE BRAKES, HYDRAULIC     4
STEERING                      3
SEATS                         3
FUEL/PROPULSION SYSTEM        3
Name: count, dtype: int64


In [9]:
power = df[df["components"].str.contains("POWER TRAIN")]
print(len(power))

for s in power["summary"].head(8):
    print(s[:200])
    print()

30
The car stopped at an traffic intersection when making a left turn. Fortunately, the traffic was light. Later, a broken driver side cv axel was identified, which had sever corrosion inside the damper.

TL* THE CONTACT OWNS A 2002 ACURA RSX TYPE S. THE CONTACT STATED THAT WHILE DRIVING APPROXIMATELY 10-60 MPH, THE THIRD GEAR SHIFTED OUT OF GEAR. IN ADDITION, THERE WAS A VERY LOUD GRINDING NOISE. THE 

2002 FORD EXPLORER XLT 4.0 LITER JERKS WHEN ENGAGING INTO REVERSE. VEHICLE JERKS HARD ENOUGH THAT THE BRAKES ARE NOT ENOUGH TO STOP IT FROM MOVING. NO OTHER GEAR IS LIKE THIS. UPON FURTHER STUDY FROM 

TL*THE CONTACT OWNS A 2002 ACURA RSX.  WHEN SWITCHING FROM SECOND TO THIRD GEAR, THE SHIFTER WOULD POP OUT OF GEAR AND MAKE A GRINDING NOISE.  THE MANUFACTURER COULD NOT ASSIST BECAUSE THERE WERE NO R

I'VE HAD TRANSMISSION PROBLEMS EVER SINCE I PURCHASED THE NEW RSX IN 02... THIS PROBLEM HAS BEEN WIDELY REPORTED ON THE INTERNET... HOWEVER ACURA HAS FAILED TO ADDRESS THIS ISSUE... THE PRO

vin is the independent identifier. A VIN is the vehicle identification number stamped on every car. These are cut to 11 characters, but that still carries a lot:
The first 3 characters identify the manufacturer. JH4 is Acura.
The 10th character is the model year. In JH4DC53052C it’s 2, which means 2002.
Characters 4 to 8 describe the model. I think DC5 is the RSX’s chassis code, but that’s from memory, so we’ll test it against the data.

In [11]:
df["vin_prefix"] = df["vin"].str.upper().str[:6]
print(df["vin_prefix"].value_counts(dropna=False).head(10))
print()

wrong = df[df["summary"].str.contains("EXPLORER|MDX|SRX", case=False, na=False)]
for _, row in wrong.iterrows():
    print(row["vin"], "|", row["summary"][:60])

vin_prefix
JH4DC5    67
          20
2HNYD1     3
PLEASE     1
JH4DD5     1
19UUA5     1
Name: count, dtype: int64

JH4DC53052C | Odometer Fraud. The contact owned a 2002 Acura SRX. The cont
 | 2002 FORD EXPLORER XLT 4.0 LITER JERKS WHEN ENGAGING INTO RE
 | MY 2002 MDX HAS 75,000 MILES AND NEEDS THE ENGINE MOUNTS REP
2HNYD18852H | MULTIPLE ENGINE MOUNT FAILURES ON 2002 ACURA MDX. FIRST FAIL
2HNYD18402H | THE 2001 ACURA MDX STALLS AT HIGH SPEEDS AND THEN KICKS BACK


In [13]:
vin = df["vin"].fillna("").str.strip().str.upper()
vin_ok = vin.str.startswith("JH4DC5")
text_ok = df["summary"].str.contains("RSX", case=False, na=False)

keep = vin_ok | ((vin == "") & text_ok)
print(keep.sum(), "kept of", len(df))
print()

for _, row in df[~keep].head(12).iterrows():
    print(row["vin"], "|", row["summary"][:60])

73 kept of 93

 | 2002 FORD EXPLORER XLT 4.0 LITER JERKS WHEN ENGAGING INTO RE
 | MY BUSINESS PURCHASED TWO BRAND NEW 2002 ACURA VEHICLES, A C
 | MY 2002 MDX HAS 75,000 MILES AND NEEDS THE ENGINE MOUNTS REP
2HNYD18892H | MIL COMES ON WITH DTC P1457 - SIGNALING EVAP BYPASS SOLENOID
2HNYD18852H | MULTIPLE ENGINE MOUNT FAILURES ON 2002 ACURA MDX. FIRST FAIL
2HNYD18402H | THE 2001 ACURA MDX STALLS AT HIGH SPEEDS AND THEN KICKS BACK
 | I PURCHASED 4 SIGMA REGENT TOURING 205/55TR16, 7E55 FROM TIR
 | DT: CONTACT WAS DRIVING ON THE FREEWAY AND HE COULDN'T STEER
 | OEM FOGLIGHT LENSES, CRACKS TOO EASILY.  *AK
 | SECOND GEAR GRINDING LED TO TRANSMISSION FAILURE.  NOW  I HA
 | IN THE 12 MONTHS I HAVE HAD MY CAR, IT HAS BEEN IN THE SHOP 
 | WHILE DRIVING  OVER BUMPS AND POTHOLES ABS LIGHT WOULD COME 


In [15]:
df["match_confidence"] = keep.map({True: "confident", False: "unverified"})
print(df["match_confidence"].value_counts())

match_confidence
confident     73
unverified    20
Name: count, dtype: int64


In [16]:
frames = []
for year in range(2002, 2007):
    r = requests.get(url, params={"make": "ACURA", "model": "RSX", "modelYear": year}, timeout=30)
    year_df = pd.DataFrame(r.json()["results"])
    year_df["model_year"] = year
    frames.append(year_df)
    print(year, len(year_df))

rsx = pd.concat(frames, ignore_index=True)
print("total:", len(rsx))

2002 93
2003 31
2004 17
2005 32
2006 24
total: 197


In [17]:
#Sucker Car Comparison
def get_complaints(make, model, years):
    frames = []
    for year in years:
        r = requests.get(url, params={"make": make, "model": model, "modelYear": year}, timeout=30)
        d = pd.DataFrame(r.json()["results"])
        d["model_year"] = year
        frames.append(d)
        print(model, year, len(d))
    return pd.concat(frames, ignore_index=True)

eclipse = get_complaints("MITSUBISHI", "ECLIPSE", range(2002, 2006))
print("total:", len(eclipse))

ECLIPSE 2002 104
ECLIPSE 2003 166
ECLIPSE 2004 14
ECLIPSE 2005 6
total: 290
